# Notebook 02 — Four backbones × two training recipes, 5-fold cross-validation

**Requires:** Notebook 01 finished (it created `02_splits/cv_splits.csv` and `01_data/groundnut_v2_cache448.zip`).
**Runtime:** GPU (`Runtime → Change runtime type → T4 GPU`).

### What this notebook answers
| Experiment | Reviewer comment |
|---|---|
| Every model is evaluated with **stratified group 5-fold CV** (near-duplicate photos never cross train/val/test); results are reported as mean ± SD over folds and as pooled out-of-fold predictions | R3-5, R4-3 (repeated experiments / CV), R4-6 (leakage), R3-1, R4-1 (one consistent accuracy value) |
| Each backbone (ResNet-50, EfficientNet-B0, EfficientNet-B4, ConvNeXt-Base) is trained with **both** recipes: **P** = the recipe of the original ResNet-50 notebook, **B** = the recipe of the original EfficientNet/ConvNeXt notebooks | R4-4 (fair comparison, contribution) |
| Checkpoint chosen on the **validation** split, results reported once on the held-out **test** fold | R3-4 (validation vs test) |
| Recipe-P models are re-tested under simulated field degradations (shadow, blur, noise, …) | R1-3a (field conditions) |

### Recipes (copied exactly from your four original notebooks)
| | Recipe **P** (proposed; ResNet-50 notebook) | Recipe **B** (baseline; EfficientNet/ConvNeXt notebooks) |
|---|---|---|
| Layers trained | last stage + new head (rest frozen) | all layers |
| Head | Linear–512–ReLU–Dropout(0.3)–Linear | Linear |
| Augmentation | H+V flip, rotation 20°, colour jitter 0.3, translate 10 %, random erasing 0.2 | H flip, rotation 15°, random resized crop 0.7–1, colour jitter 0.3, translate 10 %, grayscale 0.1 |
| Class imbalance | balanced sampler + class-weighted cross-entropy | none (label smoothing 0.2) |
| Optimiser / LR | Adam 1e-3, wd 1e-5, OneCycleLR | AdamW 1e-5, wd 1e-4, cosine warm restarts (T0 = 5, Tmult = 2) |
| Epochs / batch | 25 / 32 (B4: 16) | 25 / 32 (B4: 16) |
| Input size | 224 (B4: 380) | 224 (B4: 380) |

The full 25-epoch schedule is always run and the epoch with the best **validation macro-F1** is kept.

### Safe to interrupt
* Each finished fold is saved to `04_results/` immediately and is skipped on the next run.
* Inside a fold, a checkpoint is written every `CKPT_EVERY` epochs to **one** file per model in `05_checkpoints/`
  (overwritten each time; emptied after each fold; deleted when the model has finished all folds).
* After a disconnect: reconnect and **Runtime → Run all**.
* If `04_results/` holds results from older splits, the notebook stops and tells you (Notebook 01 archives them automatically).

**Expected time on a T4:** ≈ 4 h 15 min for all 8 configurations (measured in the first run).

**When finished:** send me `NB02_results.zip` (downloaded by the last cell; a copy is in `04_results/`).
It contains `research_log_NB02.md`: every number of this run at full precision, written by the notebook itself.

## 0. Settings

In [1]:
import os
NOTEBOOK, NB_TAG = "02_Train_Backbones_CV", "NB02"
DRIVE_ROOT  = os.environ.get("PEANUT_DRIVE_ROOT", "/content/drive/MyDrive/Peanut_Revision")
LOCAL_ROOT  = os.environ.get("PEANUT_LOCAL_ROOT", "/content/peanut")

# Order matters only for what finishes first. The four recipe-P runs are the most important.
RUN_CONFIGS = [
    "resnet50__P", "efficientnet_b0__P", "convnext_base__P", "efficientnet_b4__P",
    "resnet50__B", "efficientnet_b0__B", "convnext_base__B", "efficientnet_b4__B",
]
REPEATS      = [0]     # cross-validation repeat(s) to run (Notebook 01 made repeat 0)
FOLDS        = None    # None = all folds; e.g. [0, 1] to run only some
CKPT_EVERY   = 5       # epochs between checkpoints (one file per model, overwritten)
SAVE_WEIGHTS = "P"     # save final fold weights (fp16) to 06_models for: "P", "all" or "none"
ROBUSTNESS_FOR = "P"   # run the degradation tests for: "P", "all" or "none"
NUM_WORKERS  = 2

TEST_MODE = os.environ.get("PEANUT_TEST") == "1"   # used only for the smoke test outside Colab
if TEST_MODE:
    CKPT_EVERY, NUM_WORKERS, FOLDS = 2, 0, [0, 1]
    RUN_CONFIGS = os.environ.get("PEANUT_CONFIGS", ",".join(RUN_CONFIGS)).split(",")

## 1. Drive, GPU and environment

In [2]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import sys, json, time, shutil, zipfile, random, copy, platform, io, datetime as dt
from pathlib import Path
from dataclasses import dataclass, asdict, replace
import numpy as np, pandas as pd
import torch, torch.nn as nn, torchvision
import torchvision.transforms as T
import torchvision.models as M
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image, ImageEnhance, ImageFilter
import sklearn
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, cohen_kappa_score, confusion_matrix,
                             f1_score, precision_recall_fscore_support, roc_auc_score)
from sklearn.utils.class_weight import compute_class_weight

D = {k: Path(DRIVE_ROOT) / v for k, v in {"data": "01_data", "splits": "02_splits", "results": "04_results",
     "checkpoints": "05_checkpoints", "models": "06_models", "logs": "09_logs"}.items()}
for p in D.values(): p.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
GPU = torch.cuda.get_device_name(0) if AMP else "CPU"
if not AMP and not TEST_MODE:
    raise SystemExit("No GPU found. Runtime -> Change runtime type -> T4 GPU, then run all again.")
torch.backends.cudnn.benchmark = True

ENV = {"notebook": NOTEBOOK, "run_at": dt.datetime.now().isoformat(timespec="seconds"),
       "gpu": GPU, "python": platform.python_version(), "torch": torch.__version__,
       "torchvision": torchvision.__version__, "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
       "numpy": np.__version__, "scikit-learn": sklearn.__version__, "pillow": Image.__version__}
(D["logs"] / f"env_{NB_TAG}_{ENV['run_at'].replace(':', '-')}.json").write_text(json.dumps(ENV, indent=2))
print(json.dumps(ENV, indent=2))

Mounted at /content/drive
{
  "notebook": "02_Train_Backbones_CV",
  "run_at": "2026-10-31T13:24:32",
  "gpu": "Tesla T4",
  "python": "3.13.16",
  "torch": "2.11.0+cu130",
  "torchvision": "0.26.0+cu130",
  "cuda": "13.0",
  "cudnn": 92700,
  "numpy": "2.1.3",
  "scikit-learn": "1.6.1",
  "pillow": "11.3.0"
}


## 2. Load the index, the folds and the images
The 448-px cache zip is copied from Drive to the Colab disk, unzipped there, and all images are loaded into RAM
(≈ 1 GB). Training then never touches Drive.

In [3]:
CLASSES = ["Alternaria leaf spot", "Healthy", "Leaf spot (early and late)", "Rosette", "Rust"]
SHORT = ["Alternaria", "Healthy", "Leaf spot", "Rosette", "Rust"]

idx = pd.read_csv(D["splits"] / "index_original.csv")
cv = pd.read_csv(D["splits"] / "cv_splits.csv")
cv_meta = json.loads((D["splits"] / "cv_splits_meta.json").read_text())
print(f"{len(idx)} images | grouping: pHash <= {cv_meta['phash_thr']} and dHash <= {cv_meta['dhash_thr']}, "
      f"or {cv_meta.get('emb_model', '-')} cosine >= {cv_meta.get('emb_thr', '-')}, or combined (cos, pHash, dHash) {cv_meta.get('combo', '-')} "
      f"| {cv_meta['n_groups']} duplicate groups | {cv_meta['k']} folds")

# Guard: results in 04_results must come from the current splits.
sig_file = D["results"] / "split_signature.json"
if sig_file.exists() and json.loads(sig_file.read_text()) != cv_meta:
    raise SystemExit("04_results/ contains results trained on DIFFERENT splits.\n"
                     "Run Notebook 01 again (it archives old results automatically), then run this notebook again.")
sig_file.write_text(json.dumps(cv_meta, indent=2))

local_cache = Path(LOCAL_ROOT) / "cache448"
if not any(local_cache.rglob("*.jpg")):
    t0 = time.time()
    local_zip = Path(LOCAL_ROOT) / "groundnut_v2_cache448.zip"
    local_zip.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(D["data"] / "groundnut_v2_cache448.zip", local_zip)
    with zipfile.ZipFile(local_zip) as z:
        z.extractall(LOCAL_ROOT)
    local_zip.unlink()
    print(f"cache copied and unzipped in {time.time()-t0:.0f}s")

t0 = time.time()
IMAGES = {iid: np.asarray(Image.open(local_cache / rel).convert("RGB")) for iid, rel in zip(idx["image_id"], idx["cache_rel"])}
LABELS = dict(zip(idx["image_id"], idx["label"]))
print(f"{len(IMAGES)} images loaded into RAM in {time.time()-t0:.0f}s "
      f"({sum(a.nbytes for a in IMAGES.values())/1e9:.2f} GB)")

1720 images | grouping: pHash <= 12 and dHash <= 16, or dinov2_vits14 cosine >= 0.9694550037384033, or combined (cos, pHash, dHash) [0.95, 20, 20] | 1112 duplicate groups | 5 folds
cache copied and unzipped in 3s
1720 images loaded into RAM in 4s (1.04 GB)


## 3. Recipes and models

In [4]:
@dataclass
class Config:
    name: str
    backbone: str
    recipe: str
    aug: str = "A"
    sampler: bool = True
    class_weighted_loss: bool = True
    label_smoothing: float = 0.0
    optimizer: str = "adam"
    lr: float = 1e-3
    weight_decay: float = 1e-5
    scheduler: str = "onecycle"
    epochs: int = 25
    finetune: str = "partial"
    head: str = "mlp"
    batch_size: int = 32
    img_size: int = 224
    select_metric: str = "val_macro_f1"
    pretrained: bool = True
    split: str = "cv_splits"          # which split file in 02_splits/ (cv_splits = group-aware)
    notes: str = ""

def recipe_P(backbone, **kw):
    big = backbone == "efficientnet_b4"
    kw.setdefault("name", f"{backbone}__P")
    return Config(backbone=backbone, recipe="P", img_size=380 if big else 224, batch_size=16 if big else 32, **kw)

def recipe_B(backbone, **kw):
    big = backbone == "efficientnet_b4"
    kw.setdefault("name", f"{backbone}__B")
    return Config(backbone=backbone, recipe="B", aug="B", sampler=False, class_weighted_loss=False,
                  label_smoothing=0.2, optimizer="adamw", lr=1e-5, weight_decay=1e-4, scheduler="cosine_wr",
                  finetune="full", head="linear", img_size=380 if big else 224, batch_size=16 if big else 32, **kw)

BACKBONES = ["resnet50", "efficientnet_b0", "efficientnet_b4", "convnext_base"]
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def get_transforms(aug, size):
    if aug == "A":    # Disease_CNN_Transfer_learning_RasNet_50.ipynb
        train = T.Compose([T.Resize((size, size)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(20),
                           T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
                           T.RandomAffine(degrees=0, translate=(0.1, 0.1)), T.ToTensor(), T.RandomErasing(p=0.2),
                           T.Normalize(MEAN, STD)])
    elif aug == "B":  # Disease-EfficientNet-B0/B4, Disease-ConvNeXt-Base
        train = T.Compose([T.RandomHorizontalFlip(p=0.5), T.RandomRotation(15), T.RandomResizedCrop(size, scale=(0.7, 1.0)),
                           T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
                           T.RandomAffine(degrees=0, translate=(0.1, 0.1)), T.RandomGrayscale(p=0.1), T.ToTensor(),
                           T.Normalize(MEAN, STD)])
    elif aug == "none":
        train = T.Compose([T.Resize((size, size)), T.ToTensor(), T.Normalize(MEAN, STD)])
    else:
        raise ValueError(aug)
    test = T.Compose([T.Resize((size, size)), T.ToTensor(), T.Normalize(MEAN, STD)])
    return train, test

def build_model(cfg, num_classes=5):
    def head(in_f):
        if cfg.head == "mlp":
            return nn.Sequential(nn.Linear(in_f, 512), nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, num_classes))
        return nn.Linear(in_f, num_classes)
    b, pre = cfg.backbone, cfg.pretrained
    if b == "resnet50":
        m = M.resnet50(weights=M.ResNet50_Weights.IMAGENET1K_V1 if pre else None)
        m.fc = head(m.fc.in_features); trainable = ["layer4.", "fc."]
    elif b == "efficientnet_b0":
        m = M.efficientnet_b0(weights=M.EfficientNet_B0_Weights.IMAGENET1K_V1 if pre else None)
        m.classifier[1] = head(m.classifier[1].in_features); trainable = ["features.6.", "features.7.", "features.8.", "classifier."]
    elif b == "efficientnet_b4":
        m = M.efficientnet_b4(weights=M.EfficientNet_B4_Weights.IMAGENET1K_V1 if pre else None)
        m.classifier[1] = head(m.classifier[1].in_features); trainable = ["features.6.", "features.7.", "features.8.", "classifier."]
    elif b == "convnext_base":
        m = M.convnext_base(weights=M.ConvNeXt_Base_Weights.IMAGENET1K_V1 if pre else None)
        m.classifier[2] = head(m.classifier[2].in_features); trainable = ["features.6.", "features.7.", "classifier."]
    else:
        raise ValueError(b)
    if cfg.finetune == "partial":
        for n, p in m.named_parameters():
            p.requires_grad = any(n.startswith(t) for t in trainable)
    return m

def show_configs(configs):
    for cfg in configs:
        m = build_model(replace(cfg, pretrained=False))
        tr = sum(p.numel() for p in m.parameters() if p.requires_grad); tot = sum(p.numel() for p in m.parameters())
        print(f"{cfg.name:<36} trainable {tr/1e6:6.2f} M of {tot/1e6:6.2f} M | input {cfg.img_size} | batch {cfg.batch_size} | split {cfg.split}")
        del m

ALL_CONFIGS = {c.name: c for b in BACKBONES for c in (recipe_P(b), recipe_B(b))}
if TEST_MODE:
    ALL_CONFIGS = {k: replace(v, epochs=4, img_size=64, batch_size=8, pretrained=False) for k, v in ALL_CONFIGS.items()}
show_configs([ALL_CONFIGS[n] for n in RUN_CONFIGS])

resnet50__P                          trainable  16.02 M of  24.56 M | input 224 | batch 32 | split cv_splits
efficientnet_b0__P                   trainable   3.81 M of   4.67 M | input 224 | batch 32 | split cv_splits
convnext_base__P                     trainable  27.97 M of  88.09 M | input 224 | batch 32 | split cv_splits
efficientnet_b4__P                   trainable  14.83 M of  18.47 M | input 380 | batch 16 | split cv_splits
resnet50__B                          trainable  23.52 M of  23.52 M | input 224 | batch 32 | split cv_splits
efficientnet_b0__B                   trainable   4.01 M of   4.01 M | input 224 | batch 32 | split cv_splits
convnext_base__B                     trainable  87.57 M of  87.57 M | input 224 | batch 32 | split cv_splits
efficientnet_b4__B                   trainable  17.56 M of  17.56 M | input 380 | batch 16 | split cv_splits


In [5]:
# ---- simulated field degradations (applied to the 448-px image before resizing) ----
def _noise(img, std, seed):
    a = np.asarray(img).astype(np.float32) / 255.0
    a = a + np.random.default_rng(seed).normal(0, std, a.shape)
    return Image.fromarray((np.clip(a, 0, 1) * 255).astype(np.uint8))

def _occlude(img, frac, seed):
    rng = np.random.default_rng(seed); a = np.asarray(img).copy(); h, w = a.shape[:2]
    s = int(np.sqrt(frac) * min(h, w)); y, x = rng.integers(0, h - s), rng.integers(0, w - s)
    a[y:y + s, x:x + s] = 0
    return Image.fromarray(a)

def _shadow(img, factor, seed):
    rng = np.random.default_rng(seed); a = np.asarray(img).astype(np.float32); w = a.shape[1]
    x0 = rng.integers(0, w // 2); a[:, x0:x0 + w // 2] *= factor
    return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))

def _jpeg(img, q):
    buf = io.BytesIO(); img.save(buf, format="JPEG", quality=q); buf.seek(0)
    return Image.open(buf).convert("RGB")

PERTURBATIONS = {
    "clean":            lambda im, s: im,
    "dark_0.6":         lambda im, s: ImageEnhance.Brightness(im).enhance(0.6),
    "overexposed_1.4":  lambda im, s: ImageEnhance.Brightness(im).enhance(1.4),
    "low_contrast_0.6": lambda im, s: ImageEnhance.Contrast(im).enhance(0.6),
    "hard_shadow_0.4":  lambda im, s: _shadow(im, 0.4, s),
    "blur_sigma2":      lambda im, s: im.filter(ImageFilter.GaussianBlur(2)),
    "noise_sd0.05":     lambda im, s: _noise(im, 0.05, s),
    "jpeg_q20":         lambda im, s: _jpeg(im, 20),
    "occlusion_25pct":  lambda im, s: _occlude(im, 0.25, s),
    "rotation_45":      lambda im, s: im.rotate(45),
}

## 4. Training and evaluation of one fold (with checkpoints)

In [6]:
SPLITS = {"cv_splits": cv}            # notebooks can add more split tables (e.g. the naive split in Notebook 03)

class ImageSet(Dataset):
    def __init__(self, ids, tf, perturb=None):
        self.ids, self.tf, self.perturb = list(ids), tf, perturb
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        img = Image.fromarray(IMAGES[self.ids[i]])
        if self.perturb is not None:
            img = self.perturb(img, i)
        return self.tf(img), LABELS[self.ids[i]]

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def metrics(y, p, prob=None):
    lab = list(range(len(CLASSES)))
    pr, rc, f1, sup = precision_recall_fscore_support(y, p, labels=lab, zero_division=0)
    out = {"accuracy": accuracy_score(y, p), "n_correct": int((np.asarray(y) == np.asarray(p)).sum()), "n": int(len(y)),
           "balanced_accuracy": balanced_accuracy_score(y, p),
           "macro_precision": float(pr.mean()), "macro_recall": float(rc.mean()),
           "macro_f1": f1_score(y, p, average="macro", labels=lab, zero_division=0),
           "weighted_f1": f1_score(y, p, average="weighted", labels=lab, zero_division=0),
           "cohen_kappa": cohen_kappa_score(y, p),
           "per_class": {CLASSES[i]: {"precision": float(pr[i]), "recall": float(rc[i]), "f1": float(f1[i]),
                                      "support": int(sup[i])} for i in lab},
           "confusion_matrix": confusion_matrix(y, p, labels=lab).tolist()}
    if prob is not None and len(np.unique(y)) == len(lab):
        out["macro_auc_ovr"] = roc_auc_score(y, prob, multi_class="ovr", labels=lab)
    return {k: (float(v) if isinstance(v, (float, np.floating)) else v) for k, v in out.items()}

@torch.no_grad()
def predict(model, loader):
    model.eval(); ys, ps, probs, loss, n = [], [], [], 0.0, 0
    ce = nn.CrossEntropyLoss(reduction="sum")
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE)
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP):
            out = model(x)
        out = out.float(); loss += ce(out, y).item(); n += len(y)
        probs.append(torch.softmax(out, 1).cpu().numpy()); ps.append(out.argmax(1).cpu().numpy()); ys.append(y.cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps), np.concatenate(probs), loss / max(n, 1)

def ckpt_path(cfg): return D["checkpoints"] / f"{cfg.name}.pt"

def save_checkpoint(cfg, r, f, epoch, model, opt, sched, scaler, sampler_gen, best, best_state, best_epoch, log):
    state = {"config": cfg.name, "repeat": r, "fold": f, "epoch": epoch, "model": model.state_dict(),
             "opt": opt.state_dict(), "sched": sched.state_dict() if sched else None, "scaler": scaler.state_dict(),
             "sampler_gen": sampler_gen.get_state(), "best": best, "best_epoch": best_epoch,
             "best_state": {k: (v.half() if v.is_floating_point() else v) for k, v in best_state.items()},
             "log": log, "rng": {"py": random.getstate(), "np": np.random.get_state(), "torch": torch.get_rng_state(),
                                 "cuda": torch.cuda.get_rng_state_all() if AMP else None}}
    torch.save(state, ckpt_path(cfg))      # same file every time: overwritten in place, never sent to the Drive bin
    if os.environ.get("PEANUT_CRASH_AFTER_EPOCH") == str(epoch):   # smoke-test hook only
        raise KeyboardInterrupt("simulated disconnect")

def load_checkpoint(cfg, r, f):
    p = ckpt_path(cfg)
    if not p.exists() or p.stat().st_size == 0: return None
    try:
        st = torch.load(p, map_location="cpu", weights_only=False)
    except Exception as e:
        print("  checkpoint unreadable, starting this fold from scratch:", e); return None
    if (st["config"], st["repeat"], st["fold"]) != (cfg.name, r, f): return None
    return st

def clear_checkpoint(cfg):
    p = ckpt_path(cfg)
    if p.exists() and p.stat().st_size > 0:
        open(p, "wb").close()             # empty it (keeps one 0-byte file instead of filling the bin)

def run_fold(cfg, r, f, robustness=False, save_weights=False):
    r, f = int(r), int(f)                 # plain Python ints (numpy ints break random.seed and JSON)
    run_dir = D["results"] / cfg.name / f"r{r}_f{f}"
    if (run_dir / "metrics.json").exists():
        print(f"[{cfg.name}] repeat {r} fold {f}: already done"); return json.loads((run_dir / "metrics.json").read_text())
    run_dir.mkdir(parents=True, exist_ok=True)
    sp = SPLITS[cfg.split]
    s = sp[(sp["repeat"] == r) & (sp["fold"] == f)]
    ids = {role: s.loc[s["role"] == role, "image_id"].tolist() for role in ("train", "val", "test")}
    seed = 1000 * r + f                      # same seed for every configuration -> paired comparisons
    seed_everything(seed)
    tf_train, tf_test = get_transforms(cfg.aug, cfg.img_size)
    y_tr = np.array([LABELS[i] for i in ids["train"]])
    gen = torch.Generator().manual_seed(seed)
    if cfg.sampler:
        w = (1.0 / np.bincount(y_tr, minlength=len(CLASSES)))[y_tr]
        dl_tr = DataLoader(ImageSet(ids["train"], tf_train), batch_size=cfg.batch_size, num_workers=NUM_WORKERS,
                           sampler=WeightedRandomSampler(torch.tensor(w, dtype=torch.double), len(w), True, generator=gen),
                           pin_memory=AMP, persistent_workers=NUM_WORKERS > 0)
    else:
        dl_tr = DataLoader(ImageSet(ids["train"], tf_train), batch_size=cfg.batch_size, shuffle=True, generator=gen,
                           num_workers=NUM_WORKERS, pin_memory=AMP, persistent_workers=NUM_WORKERS > 0)
    dl_va = DataLoader(ImageSet(ids["val"], tf_test), batch_size=64, num_workers=NUM_WORKERS, pin_memory=AMP)
    dl_te = DataLoader(ImageSet(ids["test"], tf_test), batch_size=64, num_workers=NUM_WORKERS, pin_memory=AMP)

    model = build_model(cfg).to(DEVICE)
    cw = None
    if cfg.class_weighted_loss:
        cw = torch.tensor(compute_class_weight("balanced", classes=np.arange(len(CLASSES)), y=y_tr), dtype=torch.float, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=cw, label_smoothing=cfg.label_smoothing)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = (torch.optim.Adam if cfg.optimizer == "adam" else torch.optim.AdamW)(params, lr=cfg.lr, weight_decay=cfg.weight_decay)
    per_batch = cfg.scheduler == "onecycle"
    sched = {"onecycle": lambda: torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg.lr, steps_per_epoch=len(dl_tr), epochs=cfg.epochs),
             "cosine_wr": lambda: torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=5, T_mult=2),
             "cosine": lambda: torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs),
             "none": lambda: None}[cfg.scheduler]()
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)

    start, best, best_state, best_epoch, log = 0, (-1.0, -1.0), None, 0, []
    st = load_checkpoint(cfg, r, f)
    if st:
        model.load_state_dict(st["model"]); opt.load_state_dict(st["opt"]); scaler.load_state_dict(st["scaler"])
        if sched and st["sched"]: sched.load_state_dict(st["sched"])
        gen.set_state(st["sampler_gen"]); best, best_epoch, log = tuple(st["best"]), st["best_epoch"], st["log"]
        best_state = {k: v.float() if v.is_floating_point() else v for k, v in st["best_state"].items()}
        random.setstate(st["rng"]["py"]); np.random.set_state(st["rng"]["np"]); torch.set_rng_state(st["rng"]["torch"])
        if AMP and st["rng"]["cuda"]: torch.cuda.set_rng_state_all(st["rng"]["cuda"])
        start = st["epoch"]
        print(f"  resumed from checkpoint after epoch {start}")
        del st

    print(f"[{cfg.name}] repeat {r} fold {f}: train {len(ids['train'])}, val {len(ids['val'])}, test {len(ids['test'])}")
    t_fold = time.time()
    for ep in range(start, cfg.epochs):
        t0 = time.time(); model.train(); tl, tc, tn = 0.0, 0, 0
        for x, y in dl_tr:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP):
                out = model(x); loss = criterion(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            if per_batch: sched.step()
            tl += loss.item() * len(y); tc += (out.argmax(1) == y).sum().item(); tn += len(y)
        if sched is not None and not per_batch: sched.step()
        yv, pv, _, vloss = predict(model, dl_va); vm = metrics(yv, pv)
        row = {"epoch": ep + 1, "train_loss": tl / tn, "train_acc": tc / tn, "val_loss": vloss, "val_acc": vm["accuracy"],
               "val_macro_f1": vm["macro_f1"], "lr": opt.param_groups[0]["lr"], "time_s": time.time() - t0}
        log.append(row)
        score = (row[cfg.select_metric], -vloss)
        mark = ""
        if score > best:
            best, best_epoch, mark = score, ep + 1, " *"
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"  ep {ep+1:02d}  loss {row['train_loss']:.4f}  train acc {row['train_acc']:.4f}  "
              f"val acc {row['val_acc']:.4f}  val F1 {row['val_macro_f1']:.4f}  ({row['time_s']:.0f}s){mark}")
        if CKPT_EVERY and (ep + 1) % CKPT_EVERY == 0 and ep + 1 < cfg.epochs:
            save_checkpoint(cfg, r, f, ep + 1, model, opt, sched, scaler, gen, best, best_state, best_epoch, log)

    model.load_state_dict(best_state)
    yt, pt, prob, _ = predict(model, dl_te)
    m = metrics(yt, pt, prob)
    m.update({"config": asdict(cfg), "repeat": r, "fold": f, "seed": seed, "best_epoch": best_epoch,
              "best_val_macro_f1": best[0], "best_val_loss": -best[1],
              "train_minutes": (time.time() - t_fold) / 60, "n_train": len(ids["train"]), "n_val": len(ids["val"]),
              "n_test": len(ids["test"]), "gpu": GPU, "split_meta": cv_meta if cfg.split == "cv_splits" else cfg.split,
              "n_trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
              "n_total_params": sum(p.numel() for p in model.parameters())})
    pred = pd.DataFrame({"image_id": ids["test"], "y_true": yt, "y_pred": pt})
    for i in range(len(CLASSES)): pred[f"p_{i}"] = prob[:, i]

    if robustness:
        rob = {}
        for name, fn in PERTURBATIONS.items():
            dl = DataLoader(ImageSet(ids["test"], tf_test, perturb=fn), batch_size=64, num_workers=NUM_WORKERS)
            y_, p_, _, _ = predict(model, dl); mm = metrics(y_, p_)
            rob[name] = {"accuracy": mm["accuracy"], "n_correct": mm["n_correct"], "n": mm["n"], "macro_f1": mm["macro_f1"],
                         "per_class_recall": {c: mm["per_class"][c]["recall"] for c in CLASSES}}
        (run_dir / "robustness.json").write_text(json.dumps(rob, indent=2))
        m["robustness"] = {k: v["accuracy"] for k, v in rob.items()}
    if save_weights:
        wdir = D["models"] / cfg.name; wdir.mkdir(parents=True, exist_ok=True)
        torch.save({k: (v.half() if v.is_floating_point() else v) for k, v in best_state.items()}, wdir / f"r{r}_f{f}.pt")

    pd.DataFrame(log).to_csv(run_dir / "log.csv", index=False)
    pred.to_csv(run_dir / "predictions.csv", index=False)
    (run_dir / "metrics.json").write_text(json.dumps(m, indent=2))   # written last = the fold counts as done
    clear_checkpoint(cfg)
    print(f"[{cfg.name}] repeat {r} fold {f}: TEST accuracy {m['accuracy']:.4f} ({m['n_correct']}/{m['n']}), "
          f"macro-F1 {m['macro_f1']:.4f} (best epoch {best_epoch}, {m['train_minutes']:.1f} min)")
    del model, opt; torch.cuda.empty_cache() if AMP else None
    return m

def run_configs(configs, robustness_for=lambda c: False, weights_for=lambda c: False):
    for cfg in configs:
        (D["results"] / cfg.name).mkdir(parents=True, exist_ok=True)
        (D["results"] / cfg.name / "config.json").write_text(json.dumps(asdict(cfg), indent=2))
        sp = SPLITS[cfg.split]; folds_all = sorted(int(x) for x in sp["fold"].unique())
        t0 = time.time()
        for r in REPEATS:
            for f in (FOLDS if FOLDS is not None else folds_all):
                run_fold(cfg, r, f, robustness=robustness_for(cfg), save_weights=weights_for(cfg))
        done = all((D["results"] / cfg.name / f"r{r}_f{f}" / "metrics.json").exists() for r in REPEATS for f in folds_all)
        if done and ckpt_path(cfg).exists():
            ckpt_path(cfg).unlink()           # model finished: its (empty) checkpoint file is no longer needed
        print(f"==> {cfg.name} finished in {(time.time()-t0)/60:.1f} min\n")

## 5. Run
Runs every configuration in `RUN_CONFIGS`, fold by fold. Finished folds are skipped.

In [7]:
run_configs([ALL_CONFIGS[n] for n in RUN_CONFIGS],
            robustness_for=lambda c: ROBUSTNESS_FOR == "all" or (ROBUSTNESS_FOR == "P" and c.recipe == "P"),
            weights_for=lambda c: SAVE_WEIGHTS == "all" or (SAVE_WEIGHTS == "P" and c.recipe == "P"))

[resnet50__P] repeat 0 fold 0: already done
[resnet50__P] repeat 0 fold 1: already done
[resnet50__P] repeat 0 fold 2: already done
[resnet50__P] repeat 0 fold 3: already done
[resnet50__P] repeat 0 fold 4: already done
==> resnet50__P finished in 0.0 min

[efficientnet_b0__P] repeat 0 fold 0: already done
[efficientnet_b0__P] repeat 0 fold 1: already done
[efficientnet_b0__P] repeat 0 fold 2: already done
[efficientnet_b0__P] repeat 0 fold 3: already done
[efficientnet_b0__P] repeat 0 fold 4: already done
==> efficientnet_b0__P finished in 0.0 min

[convnext_base__P] repeat 0 fold 0: already done
[convnext_base__P] repeat 0 fold 1: already done
[convnext_base__P] repeat 0 fold 2: already done
[convnext_base__P] repeat 0 fold 3: already done
[convnext_base__P] repeat 0 fold 4: already done
==> convnext_base__P finished in 0.0 min

[efficientnet_b4__P] repeat 0 fold 0: already done
[efficientnet_b4__P] repeat 0 fold 1: already done
[efficientnet_b4__P] repeat 0 fold 2: already done
[eff

100%|██████████| 74.5M/74.5M [00:00<00:00, 149MB/s]


  resumed from checkpoint after epoch 20
[efficientnet_b4__B] repeat 0 fold 1: train 1255, val 147, test 318
  ep 21  loss 0.9965  train acc 0.8271  val acc 0.8912  val F1 0.7445  (70s) *
  ep 22  loss 0.9672  train acc 0.8566  val acc 0.8844  val F1 0.7701  (26s) *
  ep 23  loss 0.9448  train acc 0.8717  val acc 0.8980  val F1 0.7798  (26s) *
  ep 24  loss 0.9258  train acc 0.8741  val acc 0.9116  val F1 0.7837  (27s) *
  ep 25  loss 0.9107  train acc 0.8932  val acc 0.9048  val F1 0.7787  (28s)
[efficientnet_b4__B] repeat 0 fold 1: TEST accuracy 0.9591 (305/318), macro-F1 0.9285 (best epoch 24, 3.1 min)
[efficientnet_b4__B] repeat 0 fold 2: train 1232, val 147, test 341
  ep 01  loss 1.6034  train acc 0.2240  val acc 0.3537  val F1 0.2473  (27s) *
  ep 02  loss 1.5693  train acc 0.3523  val acc 0.4150  val F1 0.2659  (27s) *
  ep 03  loss 1.5269  train acc 0.4497  val acc 0.4898  val F1 0.3082  (27s) *
  ep 04  loss 1.4973  train acc 0.4602  val acc 0.5510  val F1 0.3361  (27s) *
  e

## 6. Summary, research log and results zip
The table is rounded for reading; the research log has the exact values.

In [8]:
def load_runs(name):
    return [json.loads(p.read_text()) for p in sorted((D["results"] / name).glob("r*_f*/metrics.json"))]

def summary_table(names):
    rows = []
    for name in names:
        ms = load_runs(name)
        if not ms: continue
        row = {"configuration": name, "folds": len(ms)}
        for k in ["accuracy", "macro_precision", "macro_recall", "macro_f1", "balanced_accuracy", "cohen_kappa"]:
            v = 100 * np.array([m[k] for m in ms])
            row[k] = f"{v.mean():.2f} ± {v.std(ddof=1) if len(v) > 1 else 0:.2f}"
        for c, sname in zip(CLASSES, SHORT):
            row[f"recall {sname}"] = f"{100*np.mean([m['per_class'][c]['recall'] for m in ms]):.1f}"
        row["pooled correct"] = f"{sum(m['n_correct'] if 'n_correct' in m else int(np.trace(m['confusion_matrix'])) for m in ms)}/{sum(m['n_test'] for m in ms)}"
        row["min/fold"] = f"{np.mean([m['train_minutes'] for m in ms]):.1f}"
        rows.append(row)
    return pd.DataFrame(rows)

def robustness_table(names):
    rows = []
    for name in names:
        ms = [m for m in load_runs(name) if "robustness" in m]
        if ms:
            rows.append({"configuration": name, **{k: f"{100*np.mean([m['robustness'][k] for m in ms]):.1f}" for k in ms[0]["robustness"]}})
    return pd.DataFrame(rows).set_index("configuration").T if rows else None

def write_research_log(names, notebook, extra_sections=()):
    # Every number is written with repr() = the exact double stored in metrics.json (no rounding).
    R = repr
    L = [f"# Research log — {notebook}", "", f"Run: {ENV['run_at']} | GPU: {ENV['gpu']} | torch {ENV['torch']} | torchvision {ENV['torchvision']}",
         f"Split rule: pHash <= {cv_meta['phash_thr']} and dHash <= {cv_meta['dhash_thr']}, or same class and {cv_meta.get('emb_model', '-')} cosine >= {cv_meta.get('emb_thr', '-')}, or same class and (cosine, pHash, dHash) within {cv_meta.get('combo', '-')}; {cv_meta['n_groups']} groups; {cv_meta['k']} folds; seed {cv_meta['seed']}",
         "", "All values below are written programmatically from metrics.json / robustness.json with full precision (Python repr).",
         "Mean and SD are computed over folds (SD with ddof = 1) from the per-fold values listed.", ""]
    keys = ["accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1", "cohen_kappa", "macro_auc_ovr"]
    for name in names:
        ms = load_runs(name)
        if not ms: continue
        L += [f"## {name}", "", "Config: `" + json.dumps(ms[0]["config"]) + "`", "",
              "| fold | n_train | n_val | n_test | correct | " + " | ".join(keys) + " | best_epoch | best_val_macro_f1 | train_minutes |",
              "|" + "---|" * (8 + len(keys)) ]
        for m in ms:
            corr = m.get("n_correct", int(np.trace(m["confusion_matrix"])))
            L.append(f"| {m['fold']} | {m['n_train']} | {m['n_val']} | {m['n_test']} | {corr} | " +
                     " | ".join(R(m[k]) if k in m else "-" for k in keys) +
                     f" | {m['best_epoch']} | {R(m.get('best_val_macro_f1', '-'))} | {R(m['train_minutes'])} |")
        L += ["", "**Across folds**", ""]
        for k in keys:
            v = [m[k] for m in ms if k in m]
            if v:
                L.append(f"- {k}: mean {R(float(np.mean(v)))}, SD {R(float(np.std(v, ddof=1))) if len(v) > 1 else '-'}, min {R(min(v))}, max {R(max(v))}")
        tot_c = sum(m.get("n_correct", int(np.trace(m["confusion_matrix"]))) for m in ms); tot_n = sum(m["n_test"] for m in ms)
        L.append(f"- pooled out-of-fold: {tot_c}/{tot_n} correct = {R(tot_c / tot_n)}")
        pooled_cm = np.sum([np.array(m["confusion_matrix"]) for m in ms], axis=0)
        L += ["", "Pooled confusion matrix (rows = true, columns = predicted; order " + ", ".join(SHORT) + "):", "```", str(pooled_cm.tolist()), "```", ""]
        L += ["Per-class (per fold: precision / recall / F1 / support):", ""]
        L.append("| fold | " + " | ".join(SHORT) + " |"); L.append("|" + "---|" * (1 + len(CLASSES)))
        for m in ms:
            L.append(f"| {m['fold']} | " + " | ".join(f"{R(m['per_class'][c]['precision'])} / {R(m['per_class'][c]['recall'])} / {R(m['per_class'][c]['f1'])} / {m['per_class'][c]['support']}" for c in CLASSES) + " |")
        L += ["", "Per-class recall across folds: " + "; ".join(f"{s}: mean {R(float(np.mean([m['per_class'][c]['recall'] for m in ms])))}" for c, s in zip(CLASSES, SHORT)), ""]
        if any("robustness" in m for m in ms):
            perts = list(ms[0]["robustness"])
            L += ["Robustness (test accuracy under each degradation, per fold):", "", "| degradation | " + " | ".join(f"fold {m['fold']}" for m in ms) + " | mean |",
                  "|" + "---|" * (2 + len(ms))]
            for pz in perts:
                v = [m["robustness"][pz] for m in ms]
                L.append(f"| {pz} | " + " | ".join(R(x) for x in v) + f" | {R(float(np.mean(v)))} |")
            L.append("")
    for title, body in extra_sections:
        L += [f"## {title}", "", body, ""]
    path = D["logs"] / f"research_log_{NB_TAG}.md"
    path.write_text("\n".join(L))
    print("research log written:", path)
    return path

def make_results_zip(names, notebook, summary_df):
    out = Path(LOCAL_ROOT) / f"{NB_TAG}_results.zip"
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for name in names:
            for p in (D["results"] / name).rglob("*"):
                if p.is_file() and p.suffix in {".json", ".csv"}:
                    z.write(p, f"04_results/{p.relative_to(D['results'])}")
        for p in D["logs"].glob(f"*{NB_TAG}*"):
            z.write(p, f"09_logs/{p.name}")
        for p in D["splits"].glob("*.json"):
            z.write(p, f"02_splits/{p.name}")
        if summary_df is not None:
            z.writestr("summary.csv", summary_df.to_csv(index=False))
    shutil.copyfile(out, D["results"] / out.name)
    print(f"Saved {D['results'] / out.name} ({out.stat().st_size/1e6:.1f} MB)")
    if IN_COLAB:
        from google.colab import files
        files.download(str(out))

SUMMARY = summary_table(RUN_CONFIGS)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
display(SUMMARY)
ROB = robustness_table(RUN_CONFIGS)
if ROB is not None:
    print("Accuracy (%) under simulated degradations (mean over folds):"); display(ROB)
write_research_log(RUN_CONFIGS, NOTEBOOK)
make_results_zip(RUN_CONFIGS, NOTEBOOK, SUMMARY)

,configuration,folds,accuracy,macro_precision,macro_recall,macro_f1,balanced_accuracy,cohen_kappa,recall Alternaria,recall Healthy,recall Leaf spot,recall Rosette,recall Rust,pooled correct,min/fold
0,resnet50__P,5,97.90 ± 0.65,97.85 ± 0.99,97.90 ± 0.87,97.83 ± 0.62,97.90 ± 0.87,97.14 ± 0.86,96.9,99.5,96.5,96.7,100.0,1684/1720,3.8
1,efficientnet_b0__P,5,97.45 ± 0.72,97.79 ± 0.63,97.74 ± 1.34,97.72 ± 0.89,97.74 ± 1.34,96.50 ± 1.03,95.2,98.6,97.1,97.8,100.0,1676/1720,3.8
2,convnext_base__P,5,98.11 ± 0.99,98.03 ± 0.55,98.24 ± 0.97,98.10 ± 0.59,98.24 ± 0.97,97.42 ± 1.34,97.3,99.9,96.3,97.8,100.0,1688/1720,5.5
3,efficientnet_b4__P,5,98.67 ± 0.50,98.74 ± 0.73,98.46 ± 1.22,98.57 ± 0.97,98.46 ± 1.22,98.18 ± 0.71,98.2,99.8,97.6,96.7,100.0,1697/1720,8.6
4,resnet50__B,5,97.95 ± 0.35,98.13 ± 0.88,98.26 ± 0.37,98.17 ± 0.52,98.26 ± 0.37,97.20 ± 0.47,96.5,100.0,95.9,98.9,100.0,1685/1720,4.8
5,efficientnet_b0__B,5,95.41 ± 0.77,96.53 ± 0.72,94.27 ± 2.63,95.27 ± 1.78,94.27 ± 2.63,93.71 ± 1.09,96.5,98.0,91.9,88.6,96.4,1641/1720,4.5
6,convnext_base__B,5,99.31 ± 0.42,99.26 ± 0.41,99.10 ± 0.74,99.16 ± 0.52,99.10 ± 0.74,99.05 ± 0.57,98.9,100.0,98.9,97.8,100.0,1708/1720,8.8
7,efficientnet_b4__B,5,94.92 ± 0.99,94.62 ± 1.07,89.78 ± 3.91,90.77 ± 4.00,89.78 ± 3.91,93.00 ± 1.47,94.2,99.7,95.3,59.8,100.0,1632/1720,10.1


Accuracy (%) under simulated degradations (mean over folds):


configuration,resnet50__P,efficientnet_b0__P,convnext_base__P,efficientnet_b4__P
clean,97.9,97.4,98.1,98.7
dark_0.6,97.0,96.4,97.8,98.7
overexposed_1.4,96.6,95.7,96.8,97.7
low_contrast_0.6,96.6,96.9,97.9,98.6
hard_shadow_0.4,94.8,93.8,96.3,98.2
blur_sigma2,96.3,95.2,97.3,97.3
noise_sd0.05,97.3,96.9,97.2,56.5
jpeg_q20,98.1,97.2,96.9,98.4
occlusion_25pct,87.9,89.6,92.1,93.2
rotation_45,97.6,97.0,98.9,98.3


research log written: /content/drive/MyDrive/Peanut_Revision/09_logs/research_log_NB02.md
Saved /content/drive/MyDrive/Peanut_Revision/04_results/NB02_results.zip (0.5 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>